In [ ]:
# Day 2 pipeline: feature selection, SMOTE, XGBoost tuning, improved NN
import os, glob
import numpy as np
import pandas as pd
from helper_functions import *
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import xgboost as xgb
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import randint, uniform

In [ ]:
# SETTINGS 
DATA_DIR = "data"
CSV_GLOB = os.path.join(DATA_DIR, "*.csv")
LABEL_COL = "Label"
RANDOM_STATE = 42
TOP_K = 50   # for mutual information selector
SMOTE_KNN = 5

In [ ]:
#  LOAD DATA 
csvs = sorted(glob.glob(CSV_GLOB))
if not csvs:
    raise FileNotFoundError("No CSVs found in data/ — download CICDDoS2019 CSV(s) into data/")
df = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True) if len(csvs)>1 else pd.read_csv(csvs[0])
print("Loaded shape:", df.shape)

In [ ]:
# basic clean (reuse your previous function)
df = basic_cleaning(df, drop_cols=['Flow ID','Timestamp','Source IP','Destination IP'])
df = df.replace([np.inf, -np.inf], np.nan)
df = simple_impute_numeric(df, strategy='median')

In [ ]:
# detect label col if different
if LABEL_COL not in df.columns:
    possible = [c for c in df.columns if 'label' in c.lower() or 'attack' in c.lower() or 'category' in c.lower()]
    if possible:
        LABEL_COL = possible[0]
        print("Using label column:", LABEL_COL)
    else:
        raise ValueError("No label column detected")

In [ ]:
# SELECT numeric features
X = df.select_dtypes(include=[np.number]).drop(columns=[LABEL_COL], errors='ignore')
y = df[LABEL_COL].astype(str)   # treat labels as strings for selector


In [ ]:
#  reduce to top-N features by mutual info
sel_k, mi_scores, mi_selected = select_kbest_mutual_info(X, y, k=TOP_K, random_state=RANDOM_STATE)
print("Top mutual-info features:", mi_selected[:10])